# 📈 informed_flow — ranking stocks by what informed participants do

A **daily, cross-sectional** strategy built on **alternative data** rather than
price alone. Each day it scores every liquid name on three independently-sourced
signals and goes long the top-ranked ones:

- **Insider buying** (SEC Form 4) — the strongest replicated alt-data signal;
  net open-market buy dollars, scaled by dollar volume.
- **Analyst rating changes** (Yahoo) — upgrade/downgrade momentum (Womack 1996);
  *changes*, never consensus levels.
- **Short interest** (FINRA) — long-only *avoidance* of high days-to-cover.

Signals are combined as an **equal-ish weighted percentile-rank composite** with
**TopK-Drop hysteresis** for turnover control. Every feature is point-in-time
(stamped `known_at`, lagged one bar) and every entry executes at the next bar's
open — no look-ahead.

Flow: **load → overview → backtest → inspect → sweep → optimize → read study →
boundary check → verify → VALIDATE (walk-forward, deflated Sharpe, cost stress,
random baseline, holdout).** Read the last section honestly.

> Data must be loaded first: run `notebooks/03_altdata_update.ipynb` (or
> `tsim altdata --dataset all --universe LiquidLeaders --since 2020-01-01`).

## 1 · Load the panel and attach alt-data features

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dataclasses import replace
from scipy import stats

from trading_simple.config import load_settings
from trading_simple.db import make_engine
from trading_simple import backtest as bt, features, jobs, optimize as opt, plots, strategies
from trading_simple.data import load_panel, symbols_with_bars

settings = load_settings()
engine = make_engine(settings)
strat = strategies.load("informed_flow")
d = strat.data

syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]), d["bar"],
                         d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
features.attach_features(panel, engine, d["features"], settings=settings)

print(f"{d['universe']}: {len(syms)} symbols, {panel.close.shape[0]} daily bars "
      f"{panel.close.index[0].date()} → {panel.close.index[-1].date()}")
for name, frame in panel.features.items():
    active = int((frame.abs() > 0).any().sum())
    print(f"  feature {name:15s}: {active}/{len(syms)} symbols with data")

## 2 · Strategy overview

The parameter table below is rendered straight from `config.yaml`, so it never
drifts from what actually runs.

In [ ]:
print("defaults (params):")
for k, v in strat.params.items():
    print(f"  {k:16s} = {v}")
print("\nsweep grid:")
for k, v in strat.sweep.items():
    print(f"  {k:16s} : {v}")
print("\noptuna search space:")
for k, v in strat.optuna["space"].items():
    print(f"  {k:16s} : {v}")

## 3 · One backtest at the defaults

Costs are always on (commission + half-spread + slippage per leg). Read
`pct_profitable` (share of names that made money) as the breadth check — a
cross-sectional strategy should win across many names, not one lucky ticker.

In [ ]:
res = bt.run_strategy(strat, panel, settings=settings)
stats0 = bt.portfolio_stats(res)
for k, v in stats0.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(res)).show()

## 4 · Inspect one name's signals

For the most insider-active name, overlay its (point-in-time) insider dollar
flow and rating changes on price — a sanity check that the features line up with
what the composite ranks on.

In [ ]:
ins = panel.features["insider"]
sym = ins.abs().sum().idxmax()
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from trading_simple.plots import SERIES

price = panel.close[sym]
cum_ins = ins[sym].fillna(0).cumsum()
rat = panel.features["ratings"][sym].fillna(0)
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=price.index, y=price, name="price", line={"color": SERIES[0]}))
fig.add_trace(go.Scatter(x=cum_ins.index, y=cum_ins, name="cumulative insider $",
                         line={"color": SERIES[2]}), secondary_y=True)
ups, downs = rat[rat > 0], rat[rat < 0]
fig.add_trace(go.Scatter(x=ups.index, y=price.reindex(ups.index), mode="markers",
                         name="upgrade",
                         marker={"color": SERIES[3], "symbol": "triangle-up", "size": 9}))
fig.add_trace(go.Scatter(x=downs.index, y=price.reindex(downs.index), mode="markers",
                         name="downgrade",
                         marker={"color": SERIES[1], "symbol": "triangle-down", "size": 9}))
fig.update_layout(title=f"{sym} — price, cumulative insider $ (right), rating changes",
                  template="plotly_white", height=420)
fig.show()

## 5 · Coarse sweep

A small grid to see the shape of the objective before the optimizer runs. The
`w_dtc: [0.0, 0.3]` column ablates short interest, so its marginal value is
visible side by side.

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
sweep = bt.sweep(strat, panel, grid, settings=settings)
display(sweep.sort_values("sharpe", ascending=False).head(10).round(3))
plots.plot_sweep(sweep).show()

## 6 · Optimize (Optuna, resumable)

Maximizes Sharpe **and** total return together (a Pareto front). The study is
journal-backed, so this cell resumes if re-run. `N_OPTIMIZE_TRIALS` is a knob —
raise it for a finer front, but remember every extra trial is another chance to
overfit (the deflated Sharpe in §10 accounts for exactly this).

In [ ]:
N_OPTIMIZE_TRIALS = 200
jobs.run_optimize("informed_flow", n_trials=N_OPTIMIZE_TRIALS)

## 7 · Read the study — admissibility before picking

Few-trade parameter sets can post a flattering Sharpe by luck, so we filter the
Pareto front to trials with at least `MIN_TRADES` before choosing (the
convention from `squeeze_breakout`). Among the admissible front we take the
Sharpe-tilted weighted best.

In [ ]:
MIN_TRADES = 100
study = opt.load_study(strat)
front = opt.pareto_frame(study)
admissible = front[front["n_trades"] >= MIN_TRADES].copy()
print(f"front: {len(front)} trials, admissible (n_trades ≥ {MIN_TRADES}): {len(admissible)}")

_INTS = ("insider_window", "rating_halflife", "max_hold", "min_events")
def coerce(params):
    """Cast integer-typed params back to int (sweep/front rows store them as float)."""
    return {k: (int(round(v)) if k in _INTS else float(v)) for k, v in params.items()}

def _norm(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0 + 1.0

pool = admissible if len(admissible) else front
pool = pool.assign(score=1.0 * _norm(pool["sharpe"]) + 0.5 * _norm(pool["total_return"]))
best_row = pool.sort_values("score", ascending=False).iloc[0]
picked = coerce({k: best_row[k] for k in strat.optuna["space"]})
print("picked params:", picked)
print("picked train metrics:", {m: round(float(best_row[m]), 4)
      for m in ("sharpe", "total_return", "pct_profitable", "n_trades")})
plots.plot_pareto(front, opt.trials_frame(study)).show()

## 8 · Boundary check

If a picked value sits on a search-space edge, the real optimum may lie outside
the box — widen that bound and re-run. (Edge flags driven only by the thin
few-trade tail are false positives; judge against the admissible front.)

In [ ]:
report = opt.boundary_report(picked, strat.optuna["space"], front=front)
display(report)

## 9 · Backtest the picked parameters

In [ ]:
res_best = bt.run_strategy(strat, panel, params=picked, settings=settings)
stats_best = bt.portfolio_stats(res_best)
for k, v in stats_best.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(res_best)).show()
display(bt.symbol_stats(res_best).sort_values("total_return", ascending=False).head(10).round(3))

## 10 · Validate — is the edge real?

Everything above is in-sample. This section is the honest test. Knobs first
(lowered here run faster; raise for a stricter check).

In [ ]:
N_RANDOM_SIMS = 200            # random-entry baseline simulations
COST_MULTIPLES = [1.0, 2.0, 3.0]  # cost-stress multiples

def slice_panel(p, start, end):
    """A copy of the panel restricted to [start, end], features sliced too."""
    m = ((p.close.index >= pd.Timestamp(start, tz="UTC"))
         & (p.close.index <= pd.Timestamp(end, tz="UTC")))
    idx = p.close.index[m]
    feats = {k: v.loc[idx] for k, v in (p.features or {}).items()}
    return replace(p, close=p.close.loc[idx], open_=p.open_.loc[idx], high=p.high.loc[idx],
                   low=p.low.loc[idx], volume=p.volume.loc[idx], features=feats)

### 10a · Costs on, then doubled — and the breakeven

Never trust a zero-cost result. Re-run the pick at 1×/2×/3× the configured
costs, then find the round-trip cost (in bps) at which the Sharpe hits zero. An
edge that dies at 2× costs was trading noise around the cost hurdle.

In [ ]:
def with_costs(**overrides):
    """A copy of settings with its (pydantic) cost block overridden."""
    return settings.model_copy(update={"costs": settings.costs.model_copy(update=overrides)})

c = settings.costs
for mult in COST_MULTIPLES:
    s = with_costs(spread_bps=c.spread_bps * mult, slippage_bps=c.slippage_bps * mult,
                   commission_per_share=c.commission_per_share * mult)
    st = bt.portfolio_stats(bt.run_strategy(strat, panel, params=picked, settings=s))
    print(f"  {mult:.0f}x costs: sharpe={st['sharpe']:.3f}  return={st['total_return']:.2%}")

base_bps = c.spread_bps + c.slippage_bps
breakeven = None
for extra in np.arange(0, 60, 2.0):
    s = with_costs(spread_bps=base_bps + extra, slippage_bps=0.0)
    st = bt.portfolio_stats(bt.run_strategy(strat, panel, params=picked, settings=s))
    if st["sharpe"] <= 0:
        breakeven = base_bps + extra
        break
print(f"\napprox breakeven half-spread: "
      f"{'>60' if breakeven is None else round(breakeven,1)} bps per leg")

### 10b · Walk-forward

Expanding-window: re-pick the best sweep config on each training slice, then
score it on the *next* unseen year. The concatenated out-of-sample Sharpes are
the number that matters — in-sample Sharpe is free.

In [ ]:
FOLDS = [("2020-06-01", "2022-12-31", "2023-01-01", "2023-06-30"),
         ("2020-06-01", "2023-06-30", "2023-07-01", "2023-12-31"),
         ("2020-06-01", "2023-12-31", "2024-01-01", "2024-06-30"),
         ("2020-06-01", "2024-06-30", "2024-07-01", "2024-12-31")]
is_sr, oos_sr = [], []
for tr0, tr1, te0, te1 in FOLDS:
    tr = bt.sweep(strat, slice_panel(panel, tr0, tr1), grid, settings=settings)
    tr = tr[tr["n_trades"] >= 30]
    row = tr.sort_values("sharpe", ascending=False).iloc[0]
    pk = coerce({k: row[k] for k in grid[0]})
    st = bt.evaluate(strat, slice_panel(panel, te0, te1), pk, settings=settings)
    is_sr.append(float(row["sharpe"]))
    oos_sr.append(float(st["sharpe"]))
    print(f"  test {te0[:7]}..{te1[:7]}: IS sharpe {row['sharpe']:.2f} → OOS {st['sharpe']:.2f}")
t, p = stats.ttest_rel(is_sr, oos_sr)
print(f"\nmean IS {np.mean(is_sr):.2f}  vs  mean OOS {np.mean(oos_sr):.2f}   "
      f"(paired t={t:.2f}, p={p:.2f}; p>0.05 = OOS not significantly worse)")

### 10c · Deflated Sharpe Ratio

The best of N optimizer trials beats a naive threshold by luck alone. The
Deflated Sharpe (Bailey & López de Prado 2014) discounts the picked Sharpe for
the number of trials tried and the return distribution's skew/kurtosis. Want
DSR > 0.95; because TPE/NSGA trials are correlated, we report it across a band
of effective trial counts.

In [ ]:
tf = opt.trials_frame(study)
trial_sr = tf["sharpe"].dropna().to_numpy()
rets = bt.equity_curve(res_best)["value"].pct_change().dropna().to_numpy()
ann = np.sqrt(252)

def deflated_sharpe(trial_sharpes, sr_ann, returns, n_eff):
    gamma = 0.5772156649
    sr_std = np.std(trial_sharpes, ddof=1)
    e_max = sr_std * ((1 - gamma) * stats.norm.ppf(1 - 1.0 / n_eff)
                      + gamma * stats.norm.ppf(1 - 1.0 / (n_eff * np.e)))
    sr, sr0 = sr_ann / ann, e_max / ann
    sk, ku = stats.skew(returns), stats.kurtosis(returns, fisher=False)
    denom = np.sqrt(1 - sk * sr + (ku - 1) / 4 * sr ** 2)
    return float(stats.norm.cdf((sr - sr0) * np.sqrt(len(returns) - 1) / denom))

sr_ann = float(best_row["sharpe"])
n = len(trial_sr)
print(f"trials with a Sharpe: {n};  picked annualized Sharpe: {sr_ann:.3f}")
for n_eff in sorted({max(2, n // 4), max(2, n // 2), n}):
    print(f"  DSR at n_eff={n_eff:4d}: {deflated_sharpe(trial_sr, sr_ann, rets, n_eff):.3f}")

### 10d · Parameter-neighborhood stability

A good optimum sits on a plateau, not a spike. Perturb each swept parameter and
require the neighborhood's *median* Sharpe to stay healthy — a sharp isolated
peak is usually overfit.

In [ ]:
nbhd = bt.sweep(strat, panel, grid, settings=settings)
adm_n = nbhd[nbhd["n_trades"] >= MIN_TRADES]["sharpe"]
print(f"neighborhood sweep: {len(nbhd)} combos, {len(adm_n)} admissible")
print(f"  sharpe  median={adm_n.median():.3f}  min={adm_n.min():.3f}  max={adm_n.max():.3f}")
print("  → plateau" if adm_n.median() > 0.5 * adm_n.max() else "  → peak (treat with caution)")
plots.plot_heatmap_grid(nbhd, pairs=[("w_insider", "q_in")],
                        metrics=["sharpe", "total_return"]).show()

### 10e · Sub-period consistency

One number over five years can hide a strategy that only worked in one regime.
Score the pick on each calendar year and read the table, not a pass/fail.

In [ ]:
years = []
for y in range(2020, 2025):
    sub = slice_panel(panel, f"{y}-01-01", f"{y}-12-31")
    st = bt.evaluate(strat, sub, picked, settings=settings)
    years.append({"year": y, **{m: round(float(st[m]), 3)
                  for m in ("total_return", "sharpe", "pct_profitable", "n_trades")}})
display(pd.DataFrame(years).set_index("year"))

### 10f · Random-entry baseline

If random entries with the same exit logic and trade count do about as well, the
signal adds nothing. The pick should beat the 95th percentile of the random
distribution.

In [ ]:
price = float(panel.close.median().median())
fees, slippage = bt.leg_costs(settings.costs, price=price)
sig = strat.signals(panel, **picked)
n_entries = max(int(sig["entries"].to_numpy().sum()), 1)
real_sr = float(bt.portfolio_stats(bt.run(panel, sig, fees=fees, slippage=slippage))["sharpe"])

rng = np.random.default_rng(0)
shape = sig["entries"].shape
sims = np.empty(N_RANDOM_SIMS)
for i in range(N_RANDOM_SIMS):
    flat = np.zeros(shape[0] * shape[1], dtype=bool)
    flat[rng.choice(flat.size, size=n_entries, replace=False)] = True
    rent = pd.DataFrame(flat.reshape(shape), index=sig["entries"].index,
                        columns=sig["entries"].columns)
    rsig = {"entries": rent, "exits": bt.exit_after(rent, picked["max_hold"])}
    sims[i] = bt.portfolio_stats(bt.run(panel, rsig, fees=fees, slippage=slippage))["sharpe"]
p95 = np.percentile(sims, 95)
pct = float((sims < real_sr).mean())
print(f"real Sharpe {real_sr:.3f}  vs  random mean {sims.mean():.3f}, p95 {p95:.3f}")
print(f"real beats {pct:.0%} of random draws → {'PASS' if real_sr > p95 else 'WEAK'} (want>p95)")

### 10g · Untouched holdout (open once)

2025–2026 was never seen by the config, the sweep, or the optimizer. Score the
pick on it exactly once. If it disappoints, that is information — do not tune
against it (that would turn the holdout into training data).

In [ ]:
full = load_panel(engine, syms, d["bar"], d["start"], None,
                  session=d.get("session", "regular"), exchanges=settings.exchanges)
features.attach_features(full, engine, d["features"], settings=settings)
holdout = slice_panel(full, "2025-01-01", str(full.close.index[-1].date()))
st = bt.portfolio_stats(bt.run_strategy(strat, holdout, params=picked, settings=settings))
print(f"HOLDOUT {holdout.close.index[0].date()} → {holdout.close.index[-1].date()} "
      f"({holdout.close.shape[0]} bars):")
for k, v in st.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(bt.run_strategy(strat, holdout, params=picked,
                                                  settings=settings))).show()

## Read the result honestly

- The **holdout** and **walk-forward** Sharpes are the real numbers; the
  in-sample and picked Sharpes are not.
- Judge breadth on **`pct_profitable`**, not one big winner, and always read
  **`n_trades`** next to any Sharpe.
- The **deflated Sharpe** and **random baseline** guard against a lucky pick
  from many trials; the **cost stress** guards against a cost-hurdle mirage.
- ~5 years is a short sample — a strong single number here is weaker evidence
  than the same number would be over 15 years. Prefer consistency across §10e
  to a high headline.
- The yfinance ratings sleeve is survivor-only; insider (SEC) and prices are
  not. Keep claims to this liquid universe.